# Домашнее задание №1, Елизавета Федоровская, группа МБРАПС261

# Задание 1 (Базовый класс Account)

In [1]:
import csv
import json
import re
from datetime import datetime

In [2]:
class Account:
    _account_counter = 1000

    def __init__(self, account_holder, balance=0):
        if not re.match(r'^[А-ЯЁA-Z][а-яёa-z]+ [А-ЯЁA-Z][а-яёa-z]+$', account_holder):
            raise ValueError("Имя должно быть в формате 'Имя Фамилия' с заглавных букв")
        
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        
        self.holder = account_holder
        self._balance = float(balance)
        self.account_number = f"ACC-{Account._account_counter}"
        Account._account_counter += 1
        self.operations_history = []
        self.rejected_rows = []

    def get_balance(self):
        return self._balance

    def get_history(self):
        return self.operations_history.copy()

    def _add_operation(self, op_type, amount, status, reason = None):
        record = {
            "operation": op_type,
            "amount": amount,
            "date": datetime.now(),
            "balance_after": self._balance,
            "status": status,
            "reason": reason
        }
        self.operations_history.append(record)

    def deposit(self, amount):
        if amount <= 0:
            self._add_operation("deposit", amount, "fail", "Сумма должна быть положительной")
            raise ValueError("Сумма должна быть положительной")
        
        self._balance += amount
        self._add_operation("deposit", amount, "success")
        return True

    def withdraw(self, amount):
        if amount <= 0:
            self._add_operation("withdraw", amount, "fail", "Сумма должна быть положительной")
            raise ValueError("Сумма должна быть положительной")
        
        if amount > self._balance:
            self._add_operation("withdraw", amount, "fail", "Недостаточно средств")
            return False
        
        self._balance -= amount
        self._add_operation("withdraw", amount, "success")
        return True

    def get_large_operations(self, n = 5, min_amount = 1000):
        large_ops = []
        for record in self.operations_history:
            if record["status"] == "success" and record["amount"] >= min_amount:
                large_ops.append(record)
        for i in range(len(large_ops)):
            for j in range(len(large_ops) - i - 1):
                if large_ops[j]["date"] < large_ops[j+1]["date"]:
                    large_ops[j], large_ops[j+1] = large_ops[j+1], large_ops[j]
        
        return large_ops[:n]

    def clean_history(self, rows, source_account_number = None):
        valid = []
        self.rejected_rows = []
        if source_account_number is None:
            source_account_number = self.account_number
        
        for i, row in enumerate(rows):
            if not isinstance(row, dict):
                self.rejected_rows.append({"row": i+1, "reason": "Не словарь"})
                continue
            if row.get("account_number") != source_account_number:
                continue
            if row.get("account_type") != self.account_type:
                self.rejected_rows.append({"row": i+1, "reason": "Неверный тип счета"})
                continue
            op = row.get("operation")
            if op not in self.allowed_operations:
                self.rejected_rows.append({"row": i+1, "reason": "Неверная операция"})
                continue
            if row.get("status") not in ("success", "fail"):
                self.rejected_rows.append({"row": i+1, "reason": "Неверный статус"})
                continue
            try:
                amount = float(row.get("amount"))
                if amount <= 0:
                    raise ValueError
            except (ValueError, TypeError):
                self.rejected_rows.append({"row": i+1, "reason": "Неверная сумма"})
                continue
            try:
                date_str = row.get("date")
                try:
                    date = datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
                except ValueError:
                    date = datetime.strptime(date_str, "%d/%m/%Y %H:%M")
            except (ValueError, TypeError):
                self.rejected_rows.append({"row": i+1, "reason": "Неверная дата"})
                continue
            valid.append({
                "operation": op,
                "amount": amount,
                "date": date,
                "balance_after": float(row.get("balance_after", 0)),
                "status": row.get("status"),
                "reason": None,
                "source_account_number": source_account_number
            })
        
        for i in range(len(valid)):
            for j in range(len(valid) - i - 1):
                if valid[j]["date"] > valid[j+1]["date"]:
                    valid[j], valid[j+1] = valid[j+1], valid[j]
        
        return valid

    def load_history(self, filename, source_account_number=None):
        if filename.endswith('.csv'):
            with open(filename, 'r', encoding ='utf-8-sig') as f:
                reader = csv.DictReader(f)
                rows = list(reader)
        elif filename.endswith('.json'):
            with open(filename, 'r', encoding ='utf-8') as f:
                rows = json.load(f)
        else:
            raise ValueError("Поддерживаются только CSV и JSON")
        
        valid = self.clean_history(rows, source_account_number)
        if valid:
            self.operations_history = valid
            for i in range(len(valid) - 1, -1, -1):
                if valid[i]["status"] == "success":
                    self._balance = valid[i]["balance_after"]
                    break
        return {"loaded": len(valid), "rejected": len(self.rejected_rows)}

# Задание №2 (этап наследования) 

In [3]:
class CheckingAccount(Account):
    account_type = "checking"
    allowed_operations = {"deposit", "withdraw"}

class SavingsAccount(Account):
    account_type = "savings"
    allowed_operations = {"deposit", "withdraw", "interest"}
    
    def withdraw(self, amount):
        if amount <= 0:
            self._add_operation("withdraw", amount, "fail", "Сумма должна быть положительной")
            raise ValueError("Сумма должна быть положительной")
        limit = self._balance / 2
        if amount > limit:
            self._add_operation("withdraw", amount, "fail", "Превышен лимит снятия (не более 50%)")
            return False
        self._balance -= amount
        self._add_operation("withdraw", amount, "success")
        return True
    
    def apply_interest(self, rate):
        if rate < 0:
            raise ValueError("Процент не может быть отрицательным")
        interest_amount = self._balance * rate / 100
        self._balance += interest_amount
        self._add_operation("interest", interest_amount, "success")
        return interest_amount

In [4]:
checking = CheckingAccount("Иван Иванов", 1000)
savings = SavingsAccount("Anna Smith", 2000)

print("Номер checking:", checking.account_number)
print("Номер savings:", savings.account_number)

checking.deposit(500)
print("Баланс после депозита:", checking.get_balance())

checking.withdraw(200)
print("Баланс после снятия:", checking.get_balance())

result = checking.withdraw(2000)
print("Попытка снять 2000:", result)

savings.withdraw(1000)
print("Баланс savings после снятия 1000:", savings.get_balance())

savings.withdraw(501)
print("Баланс savings после попытки снять 501:", savings.get_balance())

interest = savings.apply_interest(7)
print("Начисленные проценты:", interest)
print("Баланс savings после процентов:", savings.get_balance())

try:
    checking.deposit(-100)
except ValueError as e:
    print("Ошибка при отрицательном депозите:", e)

try:
    Account("иван Иванов")
except ValueError as e:
    print("Ошибка при неверном имени:", e)

print("Проверка заданий 1 и 2 пройдена")

Номер checking: ACC-1000
Номер savings: ACC-1001
Баланс после депозита: 1500.0
Баланс после снятия: 1300.0
Попытка снять 2000: False
Баланс savings после снятия 1000: 1000.0
Баланс savings после попытки снять 501: 1000.0
Начисленные проценты: 70.0
Баланс savings после процентов: 1070.0
Ошибка при отрицательном депозите: Сумма должна быть положительной
Ошибка при неверном имени: Имя должно быть в формате 'Имя Фамилия' с заглавных букв
Проверка заданий 1 и 2 пройдена


# Задание 3

In [5]:
account_csv = CheckingAccount("Петр Петров")
report_csv = account_csv.load_history('transactions_dirty.csv', 'ACC-100001')

print("Загрузка из CSV для ACC-100001:")
print("Загружено операций:", report_csv["loaded"])
print("Отклонено строк:", report_csv["rejected"])
print("Баланс после загрузки:", account_csv.get_balance())
print()
print("Примеры отклонённых строк:")
for row in account_csv.rejected_rows[:3]:
    print(row)
print()

account_json = CheckingAccount("Петр Петров")
report_json = account_json.load_history('transactions_dirty.json', 'ACC-100001')

print("Загрузка из JSON для ACC-100001:")
print("Загружено операций:", report_json["loaded"])
print("Отклонено строк:", report_json["rejected"])
print("Баланс после загрузки:", account_json.get_balance())
print()

print("Совпадают ли результаты CSV и JSON:", 
      account_csv.get_balance() == account_json.get_balance())
print()

savings_account = SavingsAccount("Anna Smith")
report_savings = savings_account.load_history('transactions_dirty.csv', 'ACC-100002')

print("Загрузка из CSV для ACC-100002 (savings):")
print("Загружено операций:", report_savings["loaded"])
print("Отклонено строк:", report_savings["rejected"])
print("Баланс после загрузки:", savings_account.get_balance())
print()

print("История операций для ACC-100001 (первые 5 записей):")
for op in account_csv.get_history()[:5]:
    print(op)

Загрузка из CSV для ACC-100001:
Загружено операций: 18
Отклонено строк: 8
Баланс после загрузки: 7959.0

Примеры отклонённых строк:
{'row': 6, 'reason': 'Неверная сумма'}
{'row': 9, 'reason': 'Неверная операция'}
{'row': 11, 'reason': 'Неверная операция'}

Загрузка из JSON для ACC-100001:
Загружено операций: 18
Отклонено строк: 8
Баланс после загрузки: 7959.0

Совпадают ли результаты CSV и JSON: True

Загрузка из CSV для ACC-100002 (savings):
Загружено операций: 19
Отклонено строк: 8
Баланс после загрузки: 5605.79

История операций для ACC-100001 (первые 5 записей):
{'operation': 'deposit', 'amount': 921.0, 'date': datetime.datetime(2025, 9, 27, 22, 17, 26), 'balance_after': 2121.0, 'status': 'success', 'reason': None, 'source_account_number': 'ACC-100001'}
{'operation': 'deposit', 'amount': 607.0, 'date': datetime.datetime(2025, 9, 27, 22, 17, 26), 'balance_after': 2728.0, 'status': 'success', 'reason': None, 'source_account_number': 'ACC-100001'}
{'operation': 'deposit', 'amount': 12